# WP16 Expanded Phase-Torus Adversary\n\nThis notebook runs a larger phase-only falsification search against the WP15 positive-stretching coefficient. It preserves modal magnitudes, polarizations, reality, and divergence freedom.\n\nThe locked benchmark from the registered WP16 run is **1.077899704172086**. A larger value makes the candidate more demanding but does not prove divergence or invalidate every finite constant.\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n!rm -rf /content/navier-stokes-bridge-audit\n!git clone -q https://github.com/reggaesharkk/navier-stokes-bridge-audit.git /content/navier-stokes-bridge-audit\n%cd /content/navier-stokes-bridge-audit\n!git checkout -q wp16-expanded-colab-phase-search-20260925\n!pip -q install numpy pandas matplotlib\n

In [ ]:
!python3 src/wp16_expanded_phase_search.py \\\n  --cutoffs 5 6 7 \\\n  --anchor-times 0.0025 0.0035 0.005 \\\n  --seeds 20260925 20260926 \\\n  --global-draws 96 \\\n  --block-rounds 6 \\\n  --block-trials 160 \\\n  --block-size 24 \\\n  --grid 24 \\\n  --output /content/wp16_expanded_phase_search_results.json\n

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt\nfrom pathlib import Path\np=Path('/content/wp16_expanded_phase_search_results.json')\ndata=json.loads(p.read_text())\nrows=[]\nfor r in data['rows']:\n    rows.append({\n        'N':r['N'],'time':r['anchor_time'],'seed':r['seed'],\n        'pairs':r['active_conjugate_pairs'],\n        'baseline':r['baseline']['C_infinity_stretch'],\n        'best':r['best']['C_infinity_stretch'],\n        'factor':r['improvement_factor'],\n        'chi':r['best']['chi_H2_high']})\ndf=pd.DataFrame(rows).sort_values('best',ascending=False)\ndisplay(df)\nprint('REGISTERED BENCHMARK =',data['registered_WP16_benchmark'])\nprint('NEW BEST =',df.iloc[0]['best'])\n

In [ ]:
plt.figure(figsize=(9,5))\nfor N in sorted(df.N.unique()):\n    g=df[df.N==N].groupby('time')['best'].max().reset_index()\n    plt.plot(g.time,g.best,marker='o',label=f'N={N}')\nplt.axhline(data['registered_WP16_benchmark'],linestyle='--',label='locked WP16 benchmark')\nplt.xlabel('evolved anchor time'); plt.ylabel('best C_infinity_stretch')\nplt.title('Expanded WP16 phase-adversary search')\nplt.grid(alpha=.3); plt.legend(); plt.show()\n

In [ ]:
outdir=Path('/content/drive/MyDrive/WP16_EXPANDED_RESULTS')\noutdir.mkdir(exist_ok=True)\n(outdir/'wp16_expanded_phase_search_results.json').write_text(p.read_text())\ndf.to_csv(outdir/'wp16_expanded_summary.csv',index=False)\nprint('Saved to',outdir)\nprint('Send me the final table or the JSON when finished.')\n